QUESTÃO III - OBSERVAÇÕES:
Peguei o dataset no Kaggle. link: 
Disponibilizo os modelos para download no meu google drive:
Professor, infelizmente não consegui pensar em nada pra possibilitar um link online com os arquivos, para que sua execução pudesse ser feita sem ter que mexer, baixar ou configurar nada. O Sr. pode fazer o download do dataset pelos links acima, caso queira executar o train, o arquivo está dentro do ZIP que disponibilizei no classroom ´´´´ task04_q3_train_vc_mestrado.ipynb ´´´.

O link para os 3 modelos (VGG, RESNET50, MOBILEV2) zipados: 

QUESTÃO III - A
Geramos o HOG,


QUESTÃO III - B
Utilizei como solicitado no enunciado da questão transferência de aprendizado com VGG16, ResNet50 e MobileNetV2 do PyTorch. Eu tentei gerar um modelo manual, na verdade consegui rodar no meu notebook localmente, mas todas as vezes não passava das 40 epochs o meu notebook fechava tudo, killaba. Mas com 37 epochs que foi até onde chegou pegou 0.95. Então criei uma instância na ec2 e gerei 3 modelos, os quais vão ser executados aqui na questão B. Para agilizar o pytorch já tem seus pesos pré-treinados em ImageNet. então me bastou definir a última camada para classificar gatos e cachorros (2), isso porque as libs já tem arquiteturas pré definidas então as camadas convolucionais são bem otimizadas, isso reflete nos resultados, poucas epochs temos altos índices de acurácia e precisão, loss constante. Apliquei uma pequena lógica pra salvar o melhor resultado, as vezes quando não usamos o stop pra pouca diferença no loss e validação, eu vou salvando o melhor modelo e pronto. Se nenhum bater, aquele é. 

In [3]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from PIL import UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchinfo import summary
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC


In [4]:
df_dir = '/Users/luryand/Documents/VC/img/task04/PetImages'

cat_files = os.listdir(os.path.join(df_dir, 'Cat'))
dog_files = os.listdir(os.path.join(df_dir, 'Dog'))

In [5]:
# limpar algumas imagens corrompidas do df
for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(df_dir, folder)
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                img.verify()
        except (IOError, SyntaxError, UnidentifiedImageError, ValueError):
            os.remove(file_path)

/Users/luryand/Documents/VC/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [8]:
# Split do dataset, train 0.7 e val 0.3
full_dataset = ImageFolder(df_dir, transform=None)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
    )

# Por Motivos de desempenho do meu notebook local em quesito cpu, vou testar o algoritmo com uma redução considerável de dados
# Torno ele ajustável através dos inputs:
train_length = 0.1
val_length = 0.1

train_limit = int(train_length * len(train_indices))
val_limit = int(val_length * len(val_indices))

limited_train_indices = train_indices.indices[:train_limit]
limited_val_indices = val_indices.indices[:val_limit]

train_paths, train_labels = [], []
val_paths, val_labels = [], []

for idx in limited_train_indices:
    path, label = full_dataset.samples[idx]
    train_paths.append(path)
    train_labels.append(label)
for idx in limited_val_indices:
    path, label = full_dataset.samples[idx]
    val_paths.append(path)
    val_labels.append(label)

In [ ]:
# Função para extrair HOG
hog = cv2.HOGDescriptor()
def extract_hog(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    return hog.compute(img).flatten()

# Recupera caminhos e labels do split já feito
train_paths, train_labels = [], []
val_paths, val_labels = [], []

for idx in train_indices.indices:
    path, label = full_dataset.samples[idx]
    train_paths.append(path)
    train_labels.append(label)
for idx in val_indices.indices:
    path, label = full_dataset.samples[idx]
    val_paths.append(path)
    val_labels.append(label)

# Extrai HOG
X_train = [extract_hog(p) for p in train_paths]
X_val = [extract_hog(p) for p in val_paths]
# Remove imagens corrompidas (None)
train_valid = [(x, y, p) for x, y, p in zip(X_train, train_labels, train_paths) if x is not None]
val_valid = [(x, y, p) for x, y, p in zip(X_val, val_labels, val_paths) if x is not None]
X_train, y_train, train_paths = zip(*train_valid)
X_val, y_val, val_paths = zip(*val_valid)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

# Treina SVM
clf = LinearSVC(max_iter=10000)
clf.fit(X_train, y_train)

# Avaliação
y_pred = clf.predict(X_val)
acc = accuracy_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)
print(f'Acurácia HOG+SVM: {acc:.2f}')
ConfusionMatrixDisplay(cm, display_labels=full_dataset.classes).plot()
plt.show()

# Exibe exemplos de acerto e erro
for i in range(5):
    idx = np.where(y_val == y_pred)[0][i]
    img = cv2.imread(val_paths[idx])
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'Real: {full_dataset.classes[y_val[idx]]} | Pred: {full_dataset.classes[y_pred[idx]]}')
    plt.axis('off')
    plt.show()

# Dei uma pesquisada e vi que o HOG é um algoritmo de detecção de objetos, então ele não é o mais adequado para classificação de imagens.
# Aparentemente temos 13 imagens que estão parcialmente corrompidas, caso queira evitar os 13 warnings, basta descomentar as linhas:
# import warnings
# warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

Corrupt JPEG data: 2230 extraneous bytes before marker 0xd9
Corrupt JPEG data: 226 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1403 extraneous bytes before marker 0xd9
Corrupt JPEG data: 162 extraneous bytes before marker 0xd9
Corrupt JPEG data: 99 extraneous bytes before marker 0xd9
Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9
Corrupt JPEG data: 128 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1153 extraneous bytes before marker 0xd9
Corrupt JPEG data: 65 extraneous bytes before marker 0xd9
Corrupt JPEG data: 254 extraneous bytes before marker 0xd9
Corrupt JPEG data: 399 extraneous bytes before marker 0xd9


In [3]:
# Augmentation do dataframe, aplicamos diversas transformações em 100% do df.
# buscamos diversificar os exemplos que damos para o modelo, criando "réplicas" modificadas das imagens originais 
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(30),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.5)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# split dos dados, 0.7 test e 0.3 val
full_dataset = ImageFolder(df_dir, transform=None)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
    )

In [4]:
def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except (UnidentifiedImageError, OSError, ValueError) as e:
        print(f"Erro ao abrir {path}: {e}")
        # Retorna uma imagem preta se der erro
        return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    
full_dataset = ImageFolder(df_dir, loader=safe_loader, transform=None)
train_dataset = ImageFolder(df_dir, loader=safe_loader, transform=train_transform)
val_dataset = ImageFolder(df_dir, loader=safe_loader, transform=val_transform)

train_sampler = SubsetRandomSampler(train_indices.indices)
val_sampler = SubsetRandomSampler(val_indices.indices)

train_loader = DataLoader(
    train_dataset, batch_size=32, sampler=train_sampler, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=32, sampler=val_sampler, num_workers=0
)

print(f"Classes: {train_dataset.classes}")
print(f"Total de amostras: {len(full_dataset)}")
print(f"Amostras de treino: {len(train_indices)}")
print(f"Amostras de validação: {len(val_indices)}")

Classes: ['Cat', 'Dog']
Total de amostras: 25000
Amostras de treino: 17500
Amostras de validação: 7500


In [5]:
# NN com 3 camadas Conv, com 2 maxpool e 2 camadas densas(Linear)
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),

    nn.Linear(64 * 28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 2)
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print({device})

model = model.to(device)

{device(type='cpu')}


In [6]:
summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [32, 2]                   --
├─Conv2d: 1-1                            [32, 16, 224, 224]        448
├─ReLU: 1-2                              [32, 16, 224, 224]        --
├─MaxPool2d: 1-3                         [32, 16, 112, 112]        --
├─Conv2d: 1-4                            [32, 32, 112, 112]        4,640
├─ReLU: 1-5                              [32, 32, 112, 112]        --
├─MaxPool2d: 1-6                         [32, 32, 56, 56]          --
├─Conv2d: 1-7                            [32, 64, 56, 56]          18,496
├─ReLU: 1-8                              [32, 64, 56, 56]          --
├─MaxPool2d: 1-9                         [32, 64, 28, 28]          --
├─Flatten: 1-10                          [32, 50176]               --
├─Linear: 1-11                           [32, 512]                 25,690,624
├─ReLU: 1-12                             [32, 512]                 --

In [7]:
# remover trash content, o DF de treinamento veio com algumas imagens corrompidas. Vamos apenas removê-las antes de run train.
def remove_corrupted_images(df_dir):
    for folder in ['Cat', 'Dog']:
        folder_path = os.path.join(df_dir, folder)
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Removendo imagem corrompida: {file_path}")
                os.remove(file_path)

In [ ]:
# CrossEntropyLoss já aplica o softmax internamente aos logits de saída do modelo.
# Por isso não está sendo chamado explicitamente no final da rede
# O modelo retorna scores brutos (os logits) e o PyTorch calcula as probabilidades automaticamente na loss.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Época {epoch+1}: Loss treino={epoch_loss:.4f} | Acc treino={epoch_acc:.4f}")

Epoch 1/50:  39%|███▉      | 214/547 [07:42<11:15,  2.03s/it]

Erro ao abrir /Users/luryand/Documents/VC/img/task04/PetImages/Dog/11702.jpg: cannot identify image file '/Users/luryand/Documents/VC/img/task04/PetImages/Dog/11702.jpg'


Epoch 1/50:  50%|████▉     | 272/547 [09:38<09:16,  2.02s/it]/Users/luryand/Documents/VC/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 1/50: 100%|██████████| 547/547 [19:01<00:00,  2.09s/it]


Época 1: Loss treino=0.6953 | Acc treino=0.5135


Epoch 2/50:  30%|███       | 166/547 [04:55<12:05,  1.91s/it]

...

Epoch 36/50:  51%|█████     | 278/547 [09:35<09:05,  2.03s/it]

Erro ao abrir /Users/luryand/Documents/VC/img/task04/PetImages/Dog/11702.jpg: cannot identify image file '/Users/luryand/Documents/VC/img/task04/PetImages/Dog/11702.jpg'


Epoch 36/50: 100%|██████████| 547/547 [19:09<00:00,  2.10s/it]


Época 36: Loss treino=0.1587 | Acc treino=0.9371


Epoch 37/50:  14%|█▎        | 75/547 [03:02<31:38,  4.02s/it]

In [ ]:
# Visualizar 24 gatos e 24 cachorros do conjunto de validação, lado a lado
model.eval()
images_cat, preds_cat = [], []
images_dog, preds_dog = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        for img, pred, label in zip(inputs.cpu(), predicted.cpu(), labels.cpu()):
            if label == 0 and len(images_cat) < 24:
                images_cat.append(img)
                preds_cat.append(pred.item())
            elif label == 1 and len(images_dog) < 24:
                images_dog.append(img)
                preds_dog.append(pred.item())
            if len(images_cat) == 24 and len(images_dog) == 24:
                break
        if len(images_cat) == 24 and len(images_dog) == 24:
            break

def denorm(img):
    img = img * 0.5 + 0.5
    return img.clamp(0, 1)

fig, axes = plt.subplots(4, 12, figsize=(18, 6))
for i in range(4):
    for j in range(12):
        ax = axes[i, j]
        if j < 6:
            idx = i * 6 + j
            if idx < len(images_cat):
                ax.imshow(denorm(images_cat[idx]).permute(1, 2, 0).numpy())
                ax.set_title(f"Cat\nPred: {'Cat' if preds_cat[idx]==0 else 'Dog'}", fontsize=7)
            else:
                ax.axis('off')
        else:
            idx = i * 6 + (j - 6)
            if idx < len(images_dog):
                ax.imshow(denorm(images_dog[idx]).permute(1, 2, 0).numpy())
                ax.set_title(f"Dog\nPred: {'Dog' if preds_dog[idx]==1 else 'Cat'}", fontsize=7)
            else:
                ax.axis('off')
        ax.axis('off')

plt.suptitle("Esquerda: Cats | Direita: Dogs (Predição do modelo)", fontsize=14)
plt.tight_layout()
plt.show()

NameError: name 'model' is not defined

In [ ]:
torch.save(model.state_dict(), 'modelo_catsdogs-v2.pth')